In [3]:
# Load environment variables from .env file
import os 
import random
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')


In [4]:
import dspy

In [5]:
lm = dspy.LM("openai/gpt-4.1-mini", temperature=1, api_key=api_key, max_tokens=32000)
dspy.configure(lm=lm)

In [6]:
import dspy
from datasets import load_dataset

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            'solution': x['solution'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]

    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[:int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 1

    return train_set, val_set, test_set

In [7]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(45, 45, 30)

In [12]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

In [13]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [ ]:
###Performance evaluation metrics using accuracy
####The answer is here some numerical value
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        return 0
    return int(correct_answer == llm_answer)

In [ ]:
import dspy

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 11.00 / 30 (36.7%): : 32it [04:20,  8.14s/it]                      

2026/01/03 08:54:47 INFO dspy.evaluate.evaluate: Average Metric: 11 / 30 (36.7%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,"First, interpret the numbers \(17_b\) and \(97_b\) in base \(b\). ...",49,✔️ [0]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"First, let's analyze the problem and organize the information: - P...",588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We have 9 players, and each chooses one flavor: chocolate (C), van...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We want to find all ordered integer pairs \((x,y)\) with \(x,y \in...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,We have 8-digit numbers composed of the digits 1 through 8 exactly...,279,✔️ [1]
5,An isosceles trapezoid has an inscribed circle tangent to each of ...,504,Given an isosceles trapezoid with an inscribed circle tangent to a...,504,✔️ [1]
6,"The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, an...",821,"We have twelve letters: A, B, C, D, E, F, G, H, I, J, K, L. They a...",271,✔️ [0]
7,Let $k$ be a real number such that the system \begin{align*} |25+2...,77,We are given a system of two equations involving a complex number ...,77,✔️ [1]
8,The parabola with equation $y = x^2 - 4$ is rotated $60^\circ$ cou...,62,"We are given the parabola \( y = x^2 - 4 \), and then we rotate it...",45,✔️ [0]
9,The $27$ cells of a $3 \times 9$ grid are filled in using the numb...,81,We have a \(3 \times 9\) grid divided into three \(3 \times 3\) bl...,95,✔️ [0]


EvaluationResult(score=36.67, results=<list of 30 results>)

So we got around 37% accuracy on this dataset.

In [10]:
test_set[1]['problem']

'On $\\triangle ABC$ points $A, D, E$, and $B$ lie in that order on side $\\overline{AB}$ with $AD = 4$, $DE = 16$, $EB = 8$. Points $A, F, G$ and $C$ lie in that order on side $\\overline{AC}$ with $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. Quadrilateral $DEGF$ has area $288$. Find the area of heptagon $AFNBCEM$.\n\n\\begin{tikzpicture}[scale=0.07, line join=round, line cap=round, >=stealth]\n\n    \\coordinate (A) at (100,100);\n\n    \\coordinate (D) at (95,80);\n    \\coordinate (F) at (130,80);\n    \\coordinate (M) at (165,80);\n\n    \\coordinate (N) at (0,50);\n    \\coordinate (E) at (87.5,50);\n    \\coordinate (G) at (175,50);\n\n    \\coordinate (B) at ($(D)!2!(E)$);\n    \\coordinate (C) at ($(F)!2!(G)$);\n    \n    \\fill[draw=black, fill=gray!20] (N) -- (E) -- (M) -- (F) -- cycle;\n    \\fill[draw=black, fill=gray!20] (N) -- (E) -- (C) -- (B) -- cycle;\n    \\fill[draw=black, fill=g